# Clustering visualization

## 0. 준비

In [ ]:
import os
import platform
import glob
import random

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from gensim.models import KeyedVectors
from sklearn.manifold import TSNE

### 폴더 경로 지정

In [ ]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
study1_dir = os.getcwd()

# model path
model_path = '\\..\\pretrained\\GoogleNews-vectors-negative300.bin' if os_system == 'Windows' else '/../pretrained/GoogleNews-vectors-negative300.bin'

# data/processed 폴더 위치 지정
processed_data_dir = study1_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')

# graph 이미지 저장할 폴더 위치 지정
graph_image_dir = study1_dir + ('\\graph\\clustering' if os_system == 'Windows' else '/graph/clustering')
# 폴더 없으면 생성
os.makedirs(graph_image_dir, exist_ok=True)


# 2차원벡터 파일 위치 지정
fitted_vectors_dir = study1_dir + ('\\loaded_vectors\\' if os_system == 'Windows' else '/loaded_vectors/')
# 폴더 없으면 생성
os.makedirs(fitted_vectors_dir, exist_ok=True)

### word2vec 모델 로딩

In [ ]:
word2vec_model = KeyedVectors.load_word2vec_format(study1_dir + model_path, binary=True)

### 데이터 로딩

In [ ]:
tbl_data = pd.read_csv(processed_data_dir + 'data_after_preprocessing.csv', encoding='ISO-8859-1')

### 단어 데이터로 벡터 만들기

In [ ]:
seed_words = ['key', 'money', 'friend']
target_words = ['money', 'friend']
n_respond_words = 30 # 하나의 시드당 30개의 단어 응답
n_subject = len(tbl_data) # 210
n_dim_of_vector = 300

# 벡터구하기
for seed_word in seed_words: # key, money, friend
    word_columns = [seed_word + str(i) for i in range(1, n_respond_words+1)] #  key1~30, money1~30, friend1~30

    for column in word_columns:
        # vector field 생성
        tbl_data[column + '_vec'] = np.empty(n_subject, dtype=object)

        # 피험자 한 명의 응답 단어들 벡터 처리
        for i_subject in range(n_subject):
            try:
                response_word = tbl_data.iloc[i_subject][column]
                if pd.isna(response_word) or len(response_word.strip()) == 0:# NaN, 값이 빈 칸 & 응답안해서 '', ' '로 저장된 경우 걸러내기
                    tbl_data[column + '_vec'][i_subject] = None
                    continue

                if isinstance(response_word, str):
                    response_word = response_word.split()
                    response_word = [response_word for response_word in response_word if response_word not in ['is', 'a','to','of','and']]
                    if len(response_word) == 0: # 앞에서 걸러져서 결과가 없으면, 넘어가기
                        tbl_data[column + '_vec'][i_subject] = None
                        continue

                    vec_word2vec = np.zeros((n_dim_of_vector, 0))  # 300차원의 빈 행렬 생성

                    for i_el in range(len(response_word)):
                        try:
                            vec_word2vec_in = word2vec_model[response_word[i_el]]
                        except:
                            vec_word2vec_in = word2vec_model[response_word[i_el].capitalize()]
                        # reshape: 벡터의 형태를 바꿔줄뿐. 300을 600 or 90으로 바꿀 순 없다.
                        vec_word2vec_in = vec_word2vec_in.reshape((n_dim_of_vector, 1))
                        vec_word2vec = np.hstack((vec_word2vec, vec_word2vec_in))  # 수평으로 벡터 쌓기
                    # 각 열(단어 벡터)에 대한 평균 계산
                    average_vector = np.mean(vec_word2vec, axis=1)
                    tbl_data[column + '_vec'][i_subject] = average_vector #38 tear30
            except:
                pass

## 1. 시각화

### 데이터 준비

In [ ]:
vectors_data = tbl_data.iloc[:, 91:]

# 아예 응답안한 피험자 행 제거
all_columns_are_none = vectors_data.isna().all(axis=1)
vectors_data = vectors_data[~all_columns_are_none]
len(vectors_data)

In [ ]:
# 타겟 단어 벡터화하기
target_word_vec = []
for target_word in target_words:
    target_word_vec.append(np.array(word2vec_model[target_word]))
target_word_vec = np.array(target_word_vec)


### 시각화 함수

In [ ]:
# 색깔 랜덤으로 생성하는 함수
def random_hex_color():
    r = lambda: np.random.randint(0, 256)
    return "#{:02X}{:02X}{:02X}".format(r(), r(), r())

In [ ]:
def get_scatter_of_vectors(num_total_subject, total_columns, response_vectors, target_vectors, target_words, lim: int, seed: str, save_path: str, is_save: bool):
    """
    num_total_subject: 총 피험자 수
    total_columns: 피험자별, 응답한 단어들의 모음. ex. { 0: [ 'tear1_vec', ... ], ...} 
    response_vectors: 피험자들이 응답한 단어들의 벡터 모음. 피험자 구분없이 차례로, 리스트에 2차원의 벡터들이 나열되어있다.
    target_vectors: 타겟 단어의 벡터 리스트. [(2,), (2,)]
    target_words: 타겟 단어의 str 리스트 ['money', 'friend']
    """
    plt.figure(figsize=(16, 16), dpi=300)

    total_words_count = 0
    for i_subject in range(num_total_subject):
        # if i_subject > 10:
        #     break
        # response words col
        valid_columns = total_columns[i_subject]
        num_response_words = len(valid_columns)

        x = []
        y = []

        for ind in range(total_words_count, (total_words_count + num_response_words)): # 이전까지 ~ 이전+현재
            x.append(response_vectors[ind][0])
            y.append(response_vectors[ind][1])
        
        # 피험자별 랜덤 색깔 추출해서 plot
        color = random_hex_color() 
        for i in range(len(x)):
            plt.scatter(x[i], y[i], c=color, s=10, alpha=0.6)#, edgecolors='white')
        
        total_words_count += num_response_words # 누적해서 카운트하기 위함

    # x축과 y축 범위 조절
    # plt.xlim(-lim, lim)
    # plt.ylim(-lim, lim)
    plt.title(f'Seed: {seed}')
    plt.grid(True)
    # # plt.show()

    if is_save:
        plt.savefig(save_path, dpi=300)


### 전체 피험자의 seed별 응답단어(30개씩)

#### -  seed별 벡터, 컬럼 준비

In [ ]:
def get_vectors_of_seed(seed_word: str):
    response_df = vectors_data[[col for col in vectors_data.columns if col.startswith(seed_word)]]

    all_subjects_tokens = []
    for i_subject in range(len(response_df)):
        vectors_of_subject = []
        for column, value in response_df.iloc[i_subject, :].items():
            if not (isinstance(value, (int, float)) and value != 0) and (value is not 0) and value is not None:
                vectors_of_subject.append((column, value)) 
        all_subjects_tokens.append(vectors_of_subject) # ex. subject_tokens[0]: 0번 피험자들의 벡터 정보들이 들어있음. [(컬럼이름, 벡터 ), ... ]
    print(len(all_subjects_tokens), len(all_subjects_tokens[0])) # 210명 피험자, 0번 피험자는 89개 답함

    # 응답 결과 전체에서 vector값만 뽑아내기. [ (300,), (300, ), ... ] 형태로 되어있음.
    vectors = np.array([vec for subject_tokens in all_subjects_tokens for column, vec in subject_tokens])
    combined_data = np.vstack((vectors, target_word_vec))
    print(combined_data.shape)

    # 피험자별로 응답 결과가 있는 컬럼값만 뽑아놓기
    total_columns = {}
    for i_subject, subject_tokens in enumerate(all_subjects_tokens):
        notnull_columns = []
        for column, vec in subject_tokens:
            notnull_columns.append(column)
        total_columns[i_subject] = notnull_columns
    
    return combined_data, total_columns

#### - tsne모델 피팅후, 차원축소된 데이터들을 npy파일로 저장

In [ ]:
def get_random_fitted_vectors_from_tsne(seed_word: str, combined_data, random_int, random_perplexity):

    # 차원축소 모델에 묶은 모든 데이터를 넣는다. 
    tsne_model = TSNE(perplexity=random_perplexity, n_components=2, n_iter=10000, random_state=random_int)
    fitted_vectors = tsne_model.fit_transform(combined_data)
    print(fitted_vectors.shape)

    # fitted_vectors 결과 저장
    result_filename = f'{fitted_vectors_dir}/fitted_vectors_{seed_word}_p{random_perplexity}_r{random_int}.npy'
    np.save(result_filename, fitted_vectors)

    print(f'perplexity: {random_perplexity}, random state: {random_int}')

    return fitted_vectors

In [ ]:
import random

for seed in seed_words:
    print(f'seed: {seed}')
    for i in range(10):
        random_int = random.randint(0, 500)
        random_perplexity = random.randint(5, 30)


        combined_data, total_columns = get_vectors_of_seed(seed_word=seed)
        fitted_vectors = get_random_fitted_vectors_from_tsne(seed_word=seed,
                                                             combined_data=combined_data,
                                                             random_int = random_int,
                                                             random_perplexity = random_perplexity)

        # 전체 데이터와 타겟 데이터의 결과 추출
        subject_new_vectors = fitted_vectors[:len(fitted_vectors)-len(target_words)]
        target_new_vectors = fitted_vectors[len(fitted_vectors)-len(target_words):]

        print(subject_new_vectors.shape, target_new_vectors.shape)

        get_scatter_of_vectors(
            num_total_subject=n_subject,
            total_columns = total_columns,
            response_vectors = subject_new_vectors,
            target_vectors = target_new_vectors,
            target_words = target_words,
            lim=1000,
            seed=seed,
            save_path=f'{graph_image_dir}/fitted_vectors_{seed}_p{random_perplexity}_r{random_int}.png',
            is_save=True
        )
